# EDA on Airport Stations and Codes Datasets

# PLEASE CLONE THIS NOTEBOOK INTO YOUR PERSONAL FOLDER

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import matplotlib.pyplot as plt
print("Welcome to the W261 final project!") 


# Know your mount
Here is the mounting for this class, your source for the original data! Remember, you only have Read access, not Write! Also, become familiar with `dbutils` the equivalent of `gcp` in DataProc

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

In [0]:
dbutils.fs.help()

# Data for the Project

For the project you will have 4 sources of data:

1. Airlines Data: This is the raw data of flights information. You have 3 months, 6 months, 1 year, and full data from 2015 to 2019. Remember the maxima: "Test, Test, Test", so a lot of testing in smaller samples before scaling up! Location of the data? `dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data/`, `dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_1y/`, etc. (Below the dbutils to get the folders)
2. Weather Data: Raw data for weather information. Same as before, we are sharing 3 months, 6 months, 1 year
3. Stations data: Extra information of the location of the different weather stations. Location `dbfs:/mnt/mids-w261/datasets_final_project_2022/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/`
4. OTPW Data: This is our joined data (We joined Airlines and Weather). This is the main dataset for your project, the previous 3 are given for reference. You can attempt your own join for Extra Credit. Location `dbfs:/mnt/mids-w261/OTPW_60M/OTPW_60M/` and more, several samples are given!

In [0]:
# Airline Data    
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/")
display(df_flights)

In [0]:
# Weather data
df_weather = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")
display(df_weather)

In [0]:
# Stations data      
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
display(df_stations)

In [0]:
# OTPW
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")
display(df_otpw)


# Example of EDA

In [0]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt

#df_weather = spark.read.parquet(f"{data_BASE_DIR}/datasets_final_project_2022/parquet_weather_data_3m/")

# Weather Data
df_weather = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")

# Airport and Airport Codes Data
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
df_airport_codes =  spark.read.format("csv").option("header","true").load('dbfs:/mnt/mids-w261/airport-codes_csv.csv')

# Flights Data
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/")

# OTPW Data
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")

# Grouping and aggregation for df_stations
grouped_stations = df_stations.groupBy('neighbor_id').agg(
    F.avg('distance_to_neighbor').alias('avg_distance_to_neighbor'),
).orderBy('avg_distance_to_neighbor')

display(grouped_stations)

# Grouping and aggregation for df_flights
grouped_flights = df_flights.groupBy('OP_UNIQUE_CARRIER').agg(
    F.avg('DEP_DELAY').alias('Avg_DEP_DELAY'),
    F.avg('ARR_DELAY').alias('Avg_ARR_DELAY'),
    F.avg('DISTANCE').alias('Avg_DISTANCE')
)

display(grouped_flights)

# Convert columns to appropriate data types
#df_weather = df_weather.withColumn("HourlyPrecipitationDouble", F.col("HourlyPrecipitation").cast("double"))
#df_weather = df_weather.withColumn("HourlyVisibilityDouble", F.col("HourlyVisibility").cast("double"))
#df_weather = df_weather.withColumn("HourlyWindSpeedDouble", F.col("HourlyWindSpeed").cast("double")).filter(F.col("HourlyWindSpeedDouble") #< 2000)

#df_weather = df_weather.withColumn("HourlyWindSpeedDouble",F.expr("try_cast(HourlyWindSpeed as double)")).filter(F.col("HourlyWindSpeedDouble") < 2000)

df_weather = (
    df_weather
    .withColumn("HourlyPrecipitationDouble", F.expr("try_cast(HourlyPrecipitation as double)"))
    .withColumn("HourlyVisibilityDouble", F.expr("try_cast(HourlyVisibility as double)"))
    .withColumn("HourlyWindSpeedDouble", F.expr("try_cast(HourlyWindSpeed as double)"))
    .filter(F.col("HourlyWindSpeedDouble") < 2000)
)

# Overlayed boxplots for df_weather
weather_cols = ['HourlyPrecipitationDouble', 'HourlyVisibilityDouble', 'HourlyWindSpeedDouble']
weather_data = df_weather.select(*weather_cols).toPandas()

plt.figure(figsize=(10, 6))
weather_data.boxplot(column=weather_cols)
plt.title('Boxplots of Weather Variables')
plt.xlabel('Weather Variables')
plt.ylabel('Values')
plt.xticks(rotation=45)
plt.show()

# EDA - Airline Data


In [0]:
# Airport and Airport Codes Data
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
df_airport_codes =  spark.read.format("csv").option("header","true").load('dbfs:/mnt/mids-w261/airport-codes_csv.csv')

In [0]:
display(df_stations)
df_stations.printSchema()
df_stations.show(5)
df_stations.count()

In [0]:
display(df_airport_codes)
df_airport_codes.printSchema()
df_airport_codes.show(5)
df_airport_codes.count()

In [0]:
grouped_stations = df_stations.groupBy('neighbor_id').agg(
    F.avg('distance_to_neighbor').alias('avg_distance_to_neighbor')
).orderBy('avg_distance_to_neighbor')

display(grouped_stations)

distance_data = grouped_stations.toPandas()

plt.figure(figsize=(8,5))
plt.hist(distance_data["avg_distance_to_neighbor"], bins=40)
plt.xlabel("Average Distance to Weather Station")
plt.ylabel("Number of Airports")
plt.title("Distribution of Weather Station Distance to Airports")
plt.show()

plt.figure(figsize=(6,5))
plt.boxplot(distance_data["avg_distance_to_neighbor"])
plt.ylabel("Average Station Distance")
plt.title("Boxplot of Airport–Weather Station Distance")
plt.show()

In [0]:
df_stations.select("distance_to_neighbor").describe().show()

df_airport_codes.groupBy("type").count().orderBy(F.desc("count")).show()

In [0]:
df_airport_codes.groupBy("iso_country").count().orderBy(F.desc("count")).show()

top_countries = (
    df_airport_codes
    .groupBy("iso_country")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
    .toPandas()
)

plt.figure(figsize=(10,6))
plt.bar(top_countries["iso_country"], top_countries["count"])

plt.title("Top 20 Countries by Number of Airports")
plt.xlabel("Country")
plt.ylabel("Number of Airports")

plt.show()

# Join Airport Codes with Stations
### This helps connect airports → weather stations.

In [0]:
airport_station = df_stations.join(
    df_airport_codes,
    df_stations.neighbor_call == df_airport_codes.ident,
    "left"
)

display(airport_station)

In [0]:
import pyspark.sql.functions as F

df_airport_codes = df_airport_codes.withColumn(
    "longitude",
    F.split(F.col("coordinates"), ",").getItem(0).cast("double")
).withColumn(
    "latitude",
    F.split(F.col("coordinates"), ",").getItem(1).cast("double")
)

df_airport_codes.select("ident","latitude","longitude").show(5)

In [0]:
airport_map = df_airport_codes.select("latitude","longitude").dropna().toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.scatter(
    airport_map["longitude"],
    airport_map["latitude"],
    s=2
)

plt.title("Geographic Distribution of Airports")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [0]:
stations_sample = df_stations.limit(2000).toPandas()

plt.figure(figsize=(10,6))

plt.scatter(
    stations_sample["neighbor_lon"],
    stations_sample["neighbor_lat"],
    s=5,
    label="Airports"
)

plt.scatter(
    stations_sample["lon"],
    stations_sample["lat"],
    s=3,
    label="Weather Stations"
)

plt.title("Airport and Weather Station Coverage")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()

In [0]:
distance_data = df_stations.select("distance_to_neighbor").toPandas()

plt.figure(figsize=(8,5))
plt.hist(distance_data["distance_to_neighbor"], bins=50)
plt.title("Distribution of Weather Station Distance to Airports")
plt.xlabel("Distance to Airport")
plt.ylabel("Frequency")
plt.show()

In [0]:
df_airport_codes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_airport_codes.columns
]).show()

In [0]:
df_airport_codes.filter(
    F.col("type") == "large_airport"
).select("ident","name","municipality").show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, DateType, TimestampType

def summarize_spark_df(df, name, path, sample_fraction=0.01, show_top_n=10):
    print(f"\n{'='*80}")
    print(f"{name.upper()} SUMMARY")
    print(f"{'='*80}")

    # Basic shape
    row_count = df.count()
    col_count = len(df.columns)
    print(f"Source Path: {path}")
    print(f"Rows: {row_count:,}")
    print(f"Columns: {col_count}")

    # Schema
    print("\nSchema:")
    df.printSchema()

    # Approximate memory usage from sample
    try:
        sample_pdf = df.sample(withReplacement=False, fraction=sample_fraction, seed=42).toPandas()
        sample_mem_bytes = sample_pdf.memory_usage(deep=True).sum()
        est_total_bytes = sample_mem_bytes / sample_fraction if sample_fraction > 0 else sample_mem_bytes
        print(f"\nEstimated Memory Usage: ~{est_total_bytes / (1024**2):,.2f} MB")
        print(f"(Estimated from a {sample_fraction:.0%} sample)")
    except Exception as e:
        print(f"\nEstimated Memory Usage: Unable to estimate ({e})")

    # Column types
    print("\nColumn Data Types:")
    for c, t in df.dtypes:
        print(f"  - {c}: {t}")

    # Missing values
    print("\nMissing Values Summary:")
    missing_exprs = [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
    missing_row = df.select(*missing_exprs).collect()[0].asDict()

    missing_summary = []
    for c in df.columns:
        miss_cnt = missing_row[c]
        miss_pct = (miss_cnt / row_count * 100) if row_count > 0 else 0
        missing_summary.append((c, miss_cnt, round(miss_pct, 2)))

    missing_summary = sorted(missing_summary, key=lambda x: x[1], reverse=True)
    print(f"{'Column':30s} {'Missing Count':>15s} {'Missing %':>12s}")
    print("-" * 60)
    for c, miss_cnt, miss_pct in missing_summary:
        print(f"{c:30s} {miss_cnt:15,} {miss_pct:12.2f}")

    # Duplicate count
    try:
        distinct_count = df.distinct().count()
        duplicate_count = row_count - distinct_count
        duplicate_pct = (duplicate_count / row_count * 100) if row_count > 0 else 0
        print("\nDuplicate Rows:")
        print(f"Distinct Rows: {distinct_count:,}")
        print(f"Duplicate Rows: {duplicate_count:,} ({duplicate_pct:.2f}%)")
    except Exception as e:
        print(f"\nDuplicate Rows: Unable to compute ({e})")

    # Numeric column stats
    numeric_cols = [
        field.name for field in df.schema.fields
        if isinstance(field.dataType, NumericType)
    ]

    if numeric_cols:
        print("\nNumeric Column Summary:")
        stats_exprs = []
        for c in numeric_cols:
            stats_exprs.extend([
                F.min(c).alias(f"{c}__min"),
                F.max(c).alias(f"{c}__max"),
                F.mean(c).alias(f"{c}__mean"),
                F.stddev(c).alias(f"{c}__stddev"),
            ])
        stats_row = df.select(*stats_exprs).collect()[0].asDict()

        for c in numeric_cols:
            print(f"\n  {c}:")
            print(f"    min    = {stats_row.get(f'{c}__min')}")
            print(f"    max    = {stats_row.get(f'{c}__max')}")
            print(f"    mean   = {stats_row.get(f'{c}__mean')}")
            print(f"    stddev = {stats_row.get(f'{c}__stddev')}")
    else:
        print("\nNumeric Column Summary: No numeric columns found")

    # Time/date columns
    datetime_cols = [
        field.name for field in df.schema.fields
        if isinstance(field.dataType, (DateType, TimestampType))
    ]

    if datetime_cols:
        print("\nTime-Based Columns:")
        for c in datetime_cols:
            bounds = df.select(
                F.min(c).alias("min_date"),
                F.max(c).alias("max_date")
            ).collect()[0]
            print(f"  - {c}: {bounds['min_date']} to {bounds['max_date']}")
    else:
        print("\nTime-Based Columns: None detected")

    # Categorical / cardinality stats
    print("\nColumn Cardinality (Distinct Counts):")
    cardinality = []
    for c in df.columns:
        try:
            distinct_vals = df.select(c).distinct().count()
            cardinality.append((c, distinct_vals))
        except Exception:
            cardinality.append((c, None))

    cardinality = sorted(
        cardinality,
        key=lambda x: (-1 if x[1] is None else -x[1])
    )
    print(f"{'Column':30s} {'Distinct Count':>15s}")
    print("-" * 48)
    for c, d in cardinality:
        print(f"{c:30s} {str(d):>15s}")

    # Top values for string columns
    string_cols = [c for c, t in df.dtypes if t == "string"]
    if string_cols:
        print("\nTop Values for String Columns:")
        for c in string_cols:
            print(f"\n  {c}:")
            df.groupBy(c).count().orderBy(F.desc("count")).show(show_top_n, truncate=False)

    print(f"\n{'='*80}\n")


# Example usage
summarize_spark_df(
    df_stations,
    name="Stations Dataset",
    path="dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/"
)

summarize_spark_df(
    df_airport_codes,
    name="Airport Codes Dataset",
    path="dbfs:/mnt/mids-w261/airport-codes_csv.csv"
)

# Exploratory Data Analysis (EDA): Stations & Airport Codes

## 1. Objective
The goal of this EDA is to understand and evaluate two supporting datasets:

- **Stations dataset (`df_stations`)**: Maps weather stations to nearby airports
  - **Path:** dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/
  - **Format:** Parquet  
  - **Description:** Contains mappings between weather stations and nearby airports, including distance information 

- **Airport codes dataset (`df_airport_codes`)**: Provides airport metadata
  - **Path:** dbfs:/mnt/mids-w261/airport-codes_csv.csv
  - **Format:** CSV  
  - **Description:** Contains global airport metadata (type, location, identifiers)

These datasets enable the integration of **weather observations with flight operations**, which is critical for modeling flight delays.

---

## 2. Stations Dataset Overview

### Distance Statistics

Descriptive statistics for the distance between stations and airports show:

| Metric | Value |
|------|------|
| Count | 5,004,169 |
| Mean | 1343.52 |
| Std Dev | 948.85 |
| Min | 0 |
| Max | 6435.97 |

- **Size**: ~5.0 million (5,004,169) rows, 12 columns  
- **Content**: Mapping between weather stations and nearby airports  

### Key Fields
- `station_id`, `neighbor_id`, `neighbor_call`  
- `neighbor_name`, `neighbor_state`  
- `distance_to_neighbor`  

### Key Observations
- Each record represents a **station–airport relationship**  
- **No missing values or duplicates**, indicating high data quality  

### Distance Analysis
- Mean distance: ~1343  
- Range: 0 to ~6436  

**Implications:**
- Some stations are **co-located with airports (distance = 0)**  
- Others are significantly farther away → may reduce **weather representativeness**  
- Distance variability introduces **potential noise in weather features**

---

## 3. Airport Codes Dataset Overview

### Airport Type Distribution

Airport types were analyzed to understand the structure of the dataset.

| Airport Type | Count |
|-------------|-------|
| small_airport | 34,808 |
| heliport | 12,028 |
| medium_airport | 4,537 |
| closed | 4,378 |
| seaplane_base | 1,030 |
| large_airport | 616 |

### Interpretation

- The majority of entries correspond to **small airports or heliports**.
- Only a small portion represent **large commercial airports**.
- Since the airline dataset focuses on commercial flights, **large and medium airports are likely the most relevant subset for analysis**.

- **Size**: ~57,000 (57,421) rows, 14 columns  
- **Content**: Global airport metadata  

### Key Fields
- `ident` (primary identifier)  
- `type` (airport type)  
- `iso_country`, `iso_region`, `municipality`  
- `iata_code`, `gps_code`  
- `latitude`, `longitude`  

### Key Observations
- Global coverage with strong concentration in the **United States (~23K (23,256) airports)**  
- Majority are **small airports or heliports**:
  - ~34K small airports  
  - ~12K heliports  
  - ~600 large airports  

**Implications:**
- Only **medium and large airports** are relevant for commercial flight analysis  
- Dataset requires **filtering for modeling relevance**

---

## 4. Geographic and Country Distribution

The dataset contains airports from many countries. The top countries include:

| Country | Airport Count |
|-------|------|
| United States | 23,256 |
| Brazil | 5,043 |
| Canada | 2,794 |
| Australia | 2,018 |
| Mexico | 1,405 |

### Interpretation

- The airport dataset has **global coverage**, but the **United States dominates**, which aligns well with the flight dataset used in this project.

---

## 5. Linking Stations with Airports

The stations dataset was joined with airport metadata using the airport identifier:


This join enriches weather stations with airport information such as:

- airport type
- location
- country
- municipality

This connection enables weather conditions recorded at stations to be associated with specific airports.

---

## 6. Missing Data Analysis

A missing-value check was performed on the airport metadata dataset.

Notable missing values include:

| Column | Missing Values |
|------|------|
| elevation_ft | 7,813 |
| municipality | 5,894 |
| gps_code | 15,860 |
| iata_code | 48,196 |
| local_code | 27,391 |

For joining datasets, **`ident` appears to be the most reliable key**.

### Stations Dataset
- **No missing values** → highly reliable  

### Airport Dataset
Significant missingness in:
- `iata_code` (~84%)  
- `local_code`, `gps_code`, `elevation_ft`, `municipality`  

**Implications:**
- Some airport metadata fields have **significant missingness**.
- The `iata_code` field has the largest number of missing values.
- This is expected because many entries correspond to small or inactive airports that do not have commercial airline codes.
- **`ident` is the most reliable join key**, not `iata_code`  
- Some metadata fields may have **limited modeling value**

---

## 7. Geographic & Structural Insights

- Global dataset with strong concentration in **North America (especially U.S.)**  
- High airport density in major aviation regions  
- Sparse coverage in remote areas  

**Implications:**
- Aligns well with **U.S.-focused flight dataset**  
- Geographic features can help capture **regional delay patterns**

---

## 8. Linking Stations with Airports

#### Join Logic: df_stations.neighbor_call == df_airport_codes.ident

### Outcome
- Enriches station data with:
  - Airport type  
  - Geographic location  
  - Country and municipality  

**Implications:**
- Enables mapping of **weather observations → airport-level features**  
- Critical for integrating weather into delay prediction models  

---

## 9. Key Findings

1. Stations dataset provides extensive mappings between airports and weather stations 
2. Weather station distance varies significantly in their distance from airports → impacts weather feature accuracy  
3. Airport dataset is **global but dominated by non-commercial airports** as only a small subset represent commercial airports  
4. High missingness in fields like `iata_code`
5. Geographic visualization confirms that airport density aligns with major aviation regions.  
6.  The `ident` field is the **most reliable key for integration with other datasets**

---

## 10. Modeling Implications

- Filter to **medium and large airports**  
- Incorporate `distance_to_neighbor` as:
  - A feature, or  
  - A filtering criterion  
- Use **`ident` for joins** instead of `iata_code`  
- Consider adding:
  - **Geographic features** (region, coordinates)  
  - **Airport-level attributes**

---

## 11. Next Steps

- Filter airport dataset for **commercial relevance**  
- Validate joins across:
  - Flights  
  - Weather stations  
  - Airport metadata  
- Integrate eather, flight, and airport datasets to build predictive features for flight delay modeling  
- Perform further EDA on:
  - Flight delays  
  - Weather–delay relationships

---

## 🚀 Summary

This EDA establishes a **strong foundation for integrating weather and airport data into flight delay modeling**, while highlighting key challenges such as:

- **Station distance variability**
- **Dataset relevance (commercial vs non-commercial airports)**
- **Missing metadata fields**

These insights directly inform **feature engineering and model design** in later phases.

